In [2]:
import sys, os, re, json, time, difflib
from collections import Counter
import pyarrow
import numpy as np, pandas as pd

PKG, COMMENTS = "package", "comments.parquet"
sys.path.insert(0, PKG)

from search_engine import SearchEngine
from reviews import ReviewSearch, split_points

eng = SearchEngine(PKG, device="cpu")      # اولین بار: ساخت BM25 حدود ۱۵ ثانیه
rs  = ReviewSearch(COMMENTS, eng)          # کل نظرات را در رم می‌آورد (حدود ۲ تا ۳ گیگ)

print("catalog:", eng.df.shape, "| comments:", rs.df.shape)
print(rs.df.columns.tolist())
print(rs.df["recommendation_status"].value_counts(dropna=False))
print(rs.df["rate"].describe())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4885.49it/s]


catalog: (98035, 12) | comments: (4176863, 10)
['id', 'product_id', 'rate', 'recommendation_status', 'is_buyer', 'likes', 'dislikes', 'body', 'advantages', 'disadvantages']
recommendation_status
recommended        2839078
<NA>                601519
no_idea             398386
not_recommended     337880
Name: count, dtype: Int64
count    4.176863e+06
mean     3.619736e+00
std      2.004010e+00
min      0.000000e+00
25%      3.000000e+00
50%      4.000000e+00
75%      5.000000e+00
max      2.500000e+03
Name: rate, dtype: float64


In [3]:
d = rs.df[["product_id", "rate", "recommendation_status"]]
flag = lambda v: (d["recommendation_status"] == v).fillna(False).astype(bool).to_numpy()

tmp = pd.DataFrame({
    "pid":  d["product_id"].to_numpy(),
    "rate": d["rate"].where(d["rate"].between(1, 5)).to_numpy(dtype="float64"),  # نمره‌ی نامعتبر حذف
    "rec":  flag("recommended"), "notr": flag("not_recommended"), "idk": flag("no_idea"),
})
st = (tmp.groupby("pid")
         .agg(n=("rec", "size"), n_rate=("rate", "count"), rate_sum=("rate", "sum"),
              rec=("rec", "sum"), notr=("notr", "sum"), idk=("idk", "sum")))
del tmp
st["n_status"]  = st["rec"] + st["notr"] + st["idk"]
st["mean_rate"] = st["rate_sum"] / st["n_rate"]
st["pct_rec"]   = st["rec"]  / st["n_status"]
st["pct_not"]   = st["notr"] / st["n_status"]
st["pct_idk"]   = st["idk"]  / st["n_status"]

P = eng.df[["id", "title_fa", "Brand", "Category1", "Category2", "Price", "Rate_cnt", "bayes_rate"]].copy()
P.columns = ["id", "title", "brand", "cat1", "cat2", "price", "rate_cnt", "bayes_rate"]
P["price_toman"] = (P["price"] / 10).astype("int64")      # قیمت کاتالوگ ریال است
for c in ["brand", "cat1", "cat2"]:
    P[c] = P[c].astype("string")
P = P.merge(st, left_on="id", right_index=True, how="inner").reset_index(drop=True)
pm = P.set_index("id")
print(P.shape)
print(P[["n", "n_status", "mean_rate", "pct_rec", "pct_not", "pct_idk"]].describe().round(3))
print(P.groupby("cat2")["n"].sum().sort_values(ascending=False).head(15))   # دسته‌های واقعی را اینجا ببین

(98035, 20)
               n   n_status  mean_rate    pct_rec    pct_not    pct_idk
count  98035.000  98035.000  98001.000  97987.000  97987.000  97987.000
mean      42.606     36.470      3.985      0.785      0.102      0.113
std       33.339     29.528      0.429      0.148      0.109      0.084
min        9.000      0.000      1.000      0.000      0.000      0.000
25%       15.000     13.000      3.740      0.700      0.017      0.057
50%       27.000     23.000      4.027      0.810      0.075      0.102
75%       68.000     58.000      4.281      0.893      0.143      0.158
max      100.000    100.000      5.000      1.000      1.000      1.000
cat2
فکری و آموزشی                   149895
شامپو مو                         77704
کرم مرطوب کننده و نرم کننده      77642
رنگ مو                           59367
لباس زیر زنانه                   55132
دفتر                             53892
خودکار و روان نویس               51476
تجهیزات کمپینگ                   46630
کاغذ                   

In [4]:
DIG = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")
def fa_norm(s):
    if not isinstance(s, str): return ""
    s = s.replace("ي", "ی").replace("ك", "ک").replace("\u200c", " ").translate(DIG)
    return re.sub(r"\s+", " ", s).strip()

NOISE = {"ندارد", "ندارم", "نداره", "ندارن", "ندارند", "هیچ", "هیچی", "نیست", "-", "."}
def is_noise(p):
    return p in NOISE or (len(p) <= 12 and (p.startswith("ندار") or p.startswith("هیچ")))

def dis_points(text):
    pts = (fa_norm(x) for x in split_points(text))
    return [p for p in pts if len(p) >= 3 and not is_noise(p)]

TAXO = {
 "کیفیت و دوام":        ["کیفیت", "جنس", "دوام", "زود خراب", "خراب", "پاره", "پارگی", "کنده", "شکست", "ضعیف"],
 "ظاهر و تفاوت با عکس": ["رنگ", "ظاهر", "عکس", "تصویر", "متفاوت", "فرق", "زشت"],
 "سایز و اندازه":       ["سایز", "اندازه", "کوچک", "بزرگ", "تنگ", "گشاد"],
 "بسته‌بندی و ارسال":   ["بسته بندی", "ارسال", "دیر", "پست", "له شده", "شکسته", "باز شده", "آسیب"],
 "اصالت و تقلبی":       ["تقلبی", "فیک", "کپی", "غیر اصل", "غیراصل", "اصل نبود", "اورجینال نبود"],
 "قیمت و ارزش خرید":    ["گران", "گرون", "قیمت", "نمیارزه", "نمی ارزه"],
 "عملکرد":              ["کار نمی", "کار نکرد", "عملکرد", "روشن نمی", "شارژ", "باتری", "صدا"],
 "گارانتی و خدمات":     ["گارانتی", "پشتیبانی", "مرجوع", "برگشت", "فروشنده"],
}
TAXO_RE = {k: re.compile("|".join(re.escape(fa_norm(w)) for w in v)) for k, v in TAXO.items()}

# ---- پیدا کردن دسته ----
CAT_SIZE = {}
for col in ["cat1", "cat2"]:
    for val, n in P.groupby(col)["n"].sum().items():
        if pd.notna(val):
            CAT_SIZE[(col, str(val))] = int(n)

def cat_mask(col, val):
    return (P[col] == val).fillna(False).to_numpy(dtype=bool)

def find_category(q):
    """ابتدا تطبیق دقیق، سپس طولانی‌ترین نام دسته‌ی منطبق و در نهایت جست‌وجوی محصول."""
    qn = fa_norm(q)
    if not qn:
        return None

    normalized = [(col, val, fa_norm(val)) for col, val in CAT_SIZE if fa_norm(val)]
    exact = [(col, val, norm) for col, val, norm in normalized if norm == qn]
    if exact:
        # در صورت یکسان بودن نام، سطح دقیق‌تر (cat2) ترجیح داده می‌شود.
        exact.sort(key=lambda x: (x[0] == "cat2", CAT_SIZE[(x[0], x[1])]), reverse=True)
        return exact[0][0], exact[0][1]

    hits = [(col, val, norm) for col, val, norm in normalized
            if norm in qn or qn in norm]
    if hits:
        # طولانی‌ترین تطبیق از انتخاب اشتباه دسته‌های عمومی‌تر جلوگیری می‌کند.
        hits.sort(key=lambda x: (len(x[2]), x[0] == "cat2", CAT_SIZE[(x[0], x[1])]), reverse=True)
        return hits[0][0], hits[0][1]

    r = eng.search(qn, k=40, parse=False)
    if r is not None and len(r) and "Category2" in r:
        cats = r["Category2"].dropna()
        if len(cats):
            cv = ("cat2", str(cats.value_counts().index[0]))
            return cv if cv in CAT_SIZE else None
    return None

# ---- نظرهای منفی ----
def negatives(pids, cap=60000):
    pids = np.asarray(list(pids), dtype=np.int64)
    r = rs.df[rs.df["product_id"].isin(pids)]
    low = r["rate"].between(1, 2).to_numpy()
    notrec = (r["recommendation_status"] == "not_recommended").fillna(False).astype(bool).to_numpy()
    r = r[low | notrec]
    total = len(r)
    if total > cap:
        r = r.sample(cap, random_state=0)
    return r, total

def complaint_report(neg, top_themes=6, top_phrases=10):
    n = len(neg)
    if n == 0:
        return dict(n_negative_analyzed=0, unclassified_share=0.0,
                    unclassified_comment_ids=[], themes=[], top_phrases=[])
    dis = [dis_points(x) for x in neg["disadvantages"]]
    body = neg["body"].map(fa_norm).tolist()
    txt = [" ، ".join(dp) if dp else b for dp, b in zip(dis, body)]
    ids = neg["id"].astype("int64").to_numpy()
    likes = pd.to_numeric(neg["likes"], errors="coerce").fillna(0).to_numpy()

    themes, any_hit = [], np.zeros(n, dtype=bool)
    for name, pat in TAXO_RE.items():
        hit = np.fromiter((bool(pat.search(t)) for t in txt), dtype=bool, count=n)
        any_hit |= hit
        if hit.any():
            idx = np.where(hit)[0]
            top = idx[np.argsort(-likes[idx])[:3]]
            themes.append(dict(theme=name, n=int(hit.sum()), share=round(float(hit.mean()), 3),
                               example_comment_ids=ids[top].tolist()))
    themes = sorted(themes, key=lambda t: -t["n"])[:top_themes]

    cnt, ex = Counter(), {}
    for cid, pts in zip(ids, dis):
        for p in set(pts):
            if len(p) <= 40:
                cnt[p] += 1
                ex.setdefault(p, [])
                if len(ex[p]) < 3: ex[p].append(int(cid))
    phrases = [dict(phrase=p, n=c, example_comment_ids=ex[p]) for p, c in cnt.most_common(top_phrases)]
    unclassified_ids = ids[~any_hit][:20].astype(int).tolist()
    return dict(n_negative_analyzed=n,
                unclassified_share=round(float(1 - any_hit.mean()), 3),
                unclassified_comment_ids=unclassified_ids,
                themes=themes, top_phrases=phrases)

def category_complaints(category):
    f = find_category(category)
    if not f: return {"error": f"دسته‌ای مشابه «{category}» پیدا نشد"}
    col, val = f
    sub = P[cat_mask(col, val)]
    neg, total = negatives(sub["id"].to_numpy())
    return dict(category=val, level=col, n_products=len(sub), n_comments_in_sample=int(sub["n"].sum()),
                n_negative_total=total, **complaint_report(neg))

CAT = P.groupby("cat2")["n"].sum().idxmax()
print("دسته‌ی تست:", CAT)
print(json.dumps(category_complaints(CAT), ensure_ascii=False, indent=1)[:2500])

دسته‌ی تست: فکری و آموزشی
{
 "category": "فکری و آموزشی",
 "level": "cat2",
 "n_products": 3388,
 "n_comments_in_sample": 149895,
 "n_negative_total": 17940,
 "n_negative_analyzed": 17940,
 "unclassified_share": 0.355,
 "unclassified_comment_ids": [
  17258722,
  30957960,
  18980736,
  15735786,
  16125318,
  51374445,
  50956996,
  17347251,
  2696703,
  2697716,
  52536422,
  33116763,
  28235360,
  54107367,
  50177383,
  51360963,
  50187725,
  49797140,
  49041022,
  50038568
 ],
 "themes": [
  {
   "theme": "کیفیت و دوام",
   "n": 7153,
   "share": 0.399,
   "example_comment_ids": [
    31999275,
    34964006,
    12103286
   ]
  },
  {
   "theme": "ظاهر و تفاوت با عکس",
   "n": 2481,
   "share": 0.138,
   "example_comment_ids": [
    35855863,
    32935410,
    28290605
   ]
  },
  {
   "theme": "قیمت و ارزش خرید",
   "n": 1697,
   "share": 0.095,
   "example_comment_ids": [
    31999275,
    34964006,
    32935410
   ]
  },
  {
   "theme": "بسته‌بندی و ارسال",
   "n": 1695,
  

In [5]:
def r3(x): return None if pd.isna(x) else round(float(x), 3)

def low_recommend_popular(category, min_status=20, top=10):
    f = find_category(category)
    if not f: return {"error": f"دسته‌ای مشابه «{category}» پیدا نشد"}
    col, val = f
    p = P[cat_mask(col, val) & (P["n_status"] >= min_status).to_numpy()]
    if len(p) < 10: return {"error": "محصول کافی در این دسته نیست", "n_eligible": len(p)}
    hi, lo = p["rate_cnt"].quantile(.75), p["pct_rec"].quantile(.25)
    res = p[(p["rate_cnt"] >= hi) & (p["pct_rec"] <= lo)].sort_values(
        ["pct_rec", "rate_cnt"], ascending=[True, False]).head(top)
    return dict(category=val, n_eligible=len(p),
                thresholds={"rate_cnt_q75": int(hi), "pct_rec_q25": round(float(lo), 3)},
                rule=f"rate_cnt >= {int(hi)} (صدک ۷۵) و pct_recommended <= {lo:.2f} (صدک ۲۵)",
                note="rate_cnt از کاتالوگ محصول و درصد پیشنهاد از وضعیت نظرات محاسبه شده است.",
                products=[dict(product_id=int(r.id), title=str(r.title).strip(), brand=None if pd.isna(r.brand) else str(r.brand),
                               rate_cnt=int(r.rate_cnt), n_comments_sample=int(r.n),
                               n_status=int(r.n_status), pct_recommended=r3(r.pct_rec),
                               pct_not_recommended=r3(r.pct_not), pct_no_idea=r3(r.pct_idk),
                               mean_rate=r3(r.mean_rate)) for r in res.itertuples()])

def brand_comparison(category, top_brands=4, min_comments=200):
    f = find_category(category)
    if not f: return {"error": f"دسته‌ای مشابه «{category}» پیدا نشد"}
    col, val = f
    p = P[cat_mask(col, val)]
    p = p[p["brand"].notna()]
    p = p[(p["brand"] != "متفرقه").fillna(False).astype(bool)]
    g = (p.groupby("brand").agg(
        n_products=("id", "size"), n_comments=("n", "sum"),
        n_rate=("n_rate", "sum"), rate_sum=("rate_sum", "sum"),
        rec=("rec", "sum"), notr=("notr", "sum"), idk=("idk", "sum"),
        n_status=("n_status", "sum")))
    g = g[(g["n_comments"] >= min_comments) & (g["n_status"] > 0) & (g["n_rate"] > 0)]
    g = g.sort_values("n_comments", ascending=False).head(top_brands)

    out = []
    for brand, r in g.iterrows():
        neg, total = negatives(p.loc[p["brand"] == brand, "id"].to_numpy(), cap=20000)
        rep = complaint_report(neg, top_themes=3, top_phrases=3)
        out.append(dict(
            brand=str(brand), n_products=int(r.n_products),
            n_comments_in_sample=int(r.n_comments), n_valid_ratings=int(r.n_rate),
            n_recommendation_status=int(r.n_status),
            mean_rate=round(float(r.rate_sum / r.n_rate), 2),
            pct_recommended=round(float(r.rec / r.n_status), 3),
            pct_not_recommended=round(float(r.notr / r.n_status), 3),
            pct_no_idea=round(float(r.idk / r.n_status), 3),
            n_negative_total=total, unclassified_share=rep["unclassified_share"],
            top_complaint_themes=rep["themes"], top_phrases=rep["top_phrases"]))
    return dict(category=val,
                note="برند «متفرقه» حذف شده؛ آمار بر اساس نمونه‌ی نظرات موجود برای محصولات است؛ میانگین امتیاز فقط از امتیازهای معتبر و درصدها فقط از وضعیت‌های معتبر محاسبه می‌شوند.",
                brands=out)

print(json.dumps(low_recommend_popular(CAT), ensure_ascii=False, indent=1)[:1800])
print(json.dumps(brand_comparison(CAT), ensure_ascii=False, indent=1)[:2500])

{
 "category": "فکری و آموزشی",
 "n_eligible": 1965,
 "thresholds": {
  "rate_cnt_q75": 174,
  "pct_rec_q25": 0.707
 },
 "rule": "rate_cnt >= 174 (صدک ۷۵) و pct_recommended <= 0.71 (صدک ۲۵)",
 "note": "rate_cnt از کاتالوگ محصول و درصد پیشنهاد از وضعیت نظرات محاسبه شده است.",
 "products": [
  {
   "product_id": 3776262,
   "title": "ست اسباب بازی ابزار کد M1-200-CF",
   "brand": "متفرقه",
   "rate_cnt": 436,
   "n_comments_sample": 100,
   "n_status": 87,
   "pct_recommended": 0.345,
   "pct_not_recommended": 0.494,
   "pct_no_idea": 0.161,
   "mean_rate": 2.724
  },
  {
   "product_id": 5426839,
   "title": "ست اسباب بازی لوازم آرایشی طرح خرگوش مدل استمپر ناخن",
   "brand": "متفرقه",
   "rate_cnt": 206,
   "n_comments_sample": 100,
   "n_status": 69,
   "pct_recommended": 0.391,
   "pct_not_recommended": 0.478,
   "pct_no_idea": 0.13,
   "mean_rate": 3.011
  },
  {
   "product_id": 2526859,
   "title": "مکعب روبیک مدل AL23",
   "brand": "متفرقه",
   "rate_cnt": 1208,
   "n_comments_sam

In [6]:
def facts(pid):
    r = pm.loc[int(pid)]
    return dict(product_id=int(pid), title=str(r["title"]).strip(),
                brand=None if pd.isna(r["brand"]) else str(r["brand"]),
                price_toman=int(r["price_toman"]), rate_cnt=int(r["rate_cnt"]),
                bayes_rate=round(float(r["bayes_rate"]), 2), mean_rate_sample=r3(r["mean_rate"]),
                n_comments_sample=int(r["n"]), pct_recommended=r3(r["pct_rec"]),
                pct_not_recommended=r3(r["pct_not"]), pct_no_idea=r3(r["pct_idk"]))

def build_pack(pid, question, k_ev=8):
    return dict(facts=facts(pid),
                pros=rs.themes(pid, "adv", 3),
                cons=rs.themes(pid, "dis", 3),
                evidence=rs.evidence(pid, question, "worth", k=k_ev))

def rule_inference(packs):
    """رتبه‌بندی همه محصولات؛ اعلام برنده فقط در صورت وجود فاصله و شواهد کافی."""
    fs = [p["facts"] for p in packs.values()]
    if len(fs) < 2:
        return dict(type="inference", winner=None, ranking=[],
                    reasoning="برای مقایسه حداقل دو محصول لازم است.", caveats="")

    def sort_key(f):
        rec = f.get("pct_recommended")
        return (float(f.get("bayes_rate") or 0),
                float(rec) if rec is not None else -1,
                int(f.get("n_comments_sample") or 0))

    ranked = sorted(fs, key=sort_key, reverse=True)
    top, second = ranked[0], ranked[1]
    top_rate, second_rate = float(top.get("bayes_rate") or 0), float(second.get("bayes_rate") or 0)
    top_rec = top.get("pct_recommended")
    second_rec = second.get("pct_recommended")
    d_rate = top_rate - second_rate
    d_rec = ((float(top_rec) - float(second_rec))
             if top_rec is not None and second_rec is not None else None)
    enough = all(int(f.get("n_comments_sample") or 0) >= 20 for f in ranked[:2])
    winner = (top["product_id"] if enough and d_rate >= 0.2 and d_rec is not None and d_rec >= 0.05
              else None)

    caveats = []
    if not enough:
        caveats.append("تعداد نظرهای نمونه برای یکی از دو محصول اول کمتر از ۲۰ است")
    if d_rec is None:
        caveats.append("درصد پیشنهاد خرید برای مقایسه کامل در دسترس نیست")
    prices = [int(f["price_toman"]) for f in fs if f.get("price_toman") is not None]
    if len(prices) >= 2:
        ratio = max(prices) / max(min(prices), 1)
        if ratio > 1.5:
            caveats.append(f"بیشترین قیمت {ratio:.1f} برابر کمترین قیمت است؛ مقایسه ارزش خرید باید با توجه به بودجه انجام شود")
    caveats.append("رتبه‌بندی بر اساس امتیاز بیزی، درصد پیشنهاد خرید و تعداد نظرات نمونه است؛ رتبه به‌تنهایی اثبات‌کننده برتری قطعی نیست.")
    ranking = [dict(rank=i + 1, product_id=f["product_id"], title=f["title"],
                    bayes_rate=f["bayes_rate"], pct_recommended=f["pct_recommended"],
                    n_comments_sample=f["n_comments_sample"], price_toman=f["price_toman"])
               for i, f in enumerate(ranked)]
    reasoning = (f"محصول {top['product_id']} طبق معیارهای عددی در رتبه اول قرار گرفته است."
                 if winner is not None else
                 "با توجه به فاصله معیارها یا تعداد شواهد، اعلام برنده قطعی قابل دفاع نیست.")
    return dict(type="inference", winner=winner, ranking=ranking,
                comparison_with_runner_up=dict(d_bayes_rate=round(d_rate, 3),
                    d_pct_recommended=None if d_rec is None else round(d_rec, 3)),
                reasoning=reasoning, caveats="؛ ".join(caveats))

def compare_products(pids, question="کدام انتخاب بهتری است و چرا؟"):
    pids = list(dict.fromkeys(int(p) for p in pids))
    if len(pids) < 2:
        raise ValueError("برای مقایسه حداقل دو شناسه محصول وارد کنید.")
    missing = [p for p in pids if p not in pm.index]
    if missing:
        raise KeyError(f"شناسه محصول در داده‌های آماده‌شده پیدا نشد: {missing}")
    packs = {p: build_pack(p, question) for p in pids}
    return dict(direct_facts=[pk["facts"] for pk in packs.values()],
                review_evidence={pid: dict(pros=pk["pros"], cons=pk["cons"], evidence=pk["evidence"])
                                 for pid, pk in packs.items()},
                inference=rule_inference(packs)), packs

# تست: دو محصول از جستجوی دوستت
top2 = eng.search("کیف دوشی زنانه", k=2)
pids = [int(x) for x in top2["id"]]
res, packs = compare_products(pids)
print(json.dumps(res, ensure_ascii=False, indent=1, default=str)[:4000])

{
 "direct_facts": [
  {
   "product_id": 4415335,
   "title": "کیف دوشی زنانه مدل BRUNCH",
   "brand": "متفرقه",
   "price_toman": 177000,
   "rate_cnt": 26,
   "bayes_rate": 3.85,
   "mean_rate_sample": 4.0,
   "n_comments_sample": 37,
   "pct_recommended": 0.769,
   "pct_not_recommended": 0.154,
   "pct_no_idea": 0.077
  },
  {
   "product_id": 1890184,
   "title": "کیف دوشی زنانه طرح پروانه",
   "brand": "متفرقه",
   "price_toman": 900000,
   "rate_cnt": 12,
   "bayes_rate": 3.5,
   "mean_rate_sample": 4.478,
   "n_comments_sample": 12,
   "pct_recommended": 0.9,
   "pct_not_recommended": 0.1,
   "pct_no_idea": 0.0
  }
 ],
 "review_evidence": {
  "4415335": {
   "pros": [
    {
     "theme": "باکیفیت",
     "n_comments": 4,
     "share": 0.8,
     "evidence_ids": [
      48736281,
      54199580,
      46669642,
      46688268
     ]
    },
    {
     "theme": "خیلی کیف بامزه و نرمیه و داخلش چند تا دفتر هم جامیشه برای بردن کلاس هایی که وسیله کم دارن خیلی خوبه",
     "n_comments": 1

In [7]:
import httpx
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

client = OpenAI(base_url="https://193.93.169.12/v1", api_key=os.environ["QBC13_API_KEY"],
                http_client=httpx.Client(verify=False, timeout=120))
LOG = []
def llm(messages, tag, model="openai/gpt-4.1-mini", **kw):
    t = time.time()
    r = client.chat.completions.create(model=model, messages=messages, **kw)
    LOG.append(dict(tag=tag, model=model, sec=round(time.time() - t, 2),
                    prompt_tokens=r.usage.prompt_tokens, completion_tokens=r.usage.completion_tokens))
    return r

r = llm([{"role": "user", "content": "سلام"}], "test", max_tokens=20)
print(r.choices[0].message.content, LOG[-1])

سلام! چطوری؟ چطور می‌توانم کمکتان کنم؟ {'tag': 'test', 'model': 'openai/gpt-4.1-mini', 'sec': 2.16, 'prompt_tokens': 8, 'completion_tokens': 17}


In [8]:
SYS_COMPARE = """تو دستیار مقایسه‌ی محصولات یک فروشگاه اینترنتی هستی. فقط از داده‌ی JSON کاربر استفاده کن.
1. direct_facts: فقط اعداد و اطلاعات موجود در facts.
2. review_evidence: ادعاهای مربوط به تجربه‌ی کاربران؛ هر ادعا باید حداقل یک comment_id از شواهد همان product_id داشته باشد.
3. inference: نتیجه‌ی عددی rule_inference مرجع تصمیم‌گیری است. آن را تغییر نده و برنده‌ی دیگری انتخاب نکن. اگر winner برابر null است، تو هم برنده قطعی اعلام نکن.
4. رتبه‌بندی فقط رتبه‌ی معیارهای موجود است و به‌تنهایی اثبات‌کننده برتری قطعی نیست.
5. چیزی که در داده نیست را حدس نزن. تعداد کم نظرات و اختلاف زیاد قیمت را در caveats بگو.
خروجی فقط JSON فارسی:
{"direct_facts":[{"product_id":0,"claim":""}],
 "review_evidence":[{"product_id":0,"claim":"","comment_ids":[0]}],
 "inference":{"winner":null,"reasoning":"","caveats":""}}"""

def compare_llm(pids, question="کدام انتخاب بهتری است و چرا؟", model="openai/gpt-4.1-mini"):
    pids = list(dict.fromkeys(int(p) for p in pids))
    if len(pids) < 2:
        raise ValueError("برای مقایسه حداقل دو محصول لازم است.")
    packs = {p: build_pack(p, question) for p in pids}
    rule_result = rule_inference(packs)

    user_payload = {"question": question, "products": packs,
                    "rule_based_inference": rule_result}
    r = llm([{"role": "system", "content": SYS_COMPARE},
             {"role": "user", "content": json.dumps(user_payload, ensure_ascii=False, default=str)}],
            "compare", model=model, response_format={"type": "json_object"}, temperature=0)
    out = json.loads(r.choices[0].message.content)

    # شناسه‌های مجاز را جداگانه برای هر محصول نگه می‌داریم تا شاهد محصول A
    # به اشتباه برای ادعایی درباره محصول B استفاده نشود.
    allowed_by_product = {}
    for pid, pk in packs.items():
        allowed = set()
        for e in pk["evidence"]:
            if "comment_id" in e:
                allowed.add(str(e["comment_id"]))
        for theme in (pk["pros"] or []) + (pk["cons"] or []):
            allowed.update(str(i) for i in theme.get("evidence_ids", []))
        allowed_by_product[str(pid)] = allowed

    invalid, cross_product = [], []
    used = []
    for item in out.get("review_evidence", []):
        pid = str(item.get("product_id"))
        for cid in item.get("comment_ids", []):
            cid = str(cid)
            used.append(cid)
            if pid not in allowed_by_product or cid not in allowed_by_product.get(pid, set()):
                invalid.append({"product_id": pid, "comment_id": cid})
                if any(cid in ids for other_pid, ids in allowed_by_product.items() if other_pid != pid):
                    cross_product.append({"product_id": pid, "comment_id": cid})

    # نتیجه‌ی عددی را در خروجی نهایی نیز حفظ می‌کنیم تا اختلاف دو لایه قابل مشاهده باشد.
    out["_rule_based_inference"] = rule_result
    out["_grounding"] = dict(
        n_cited=len(used), n_invalid=len(invalid), invalid_citations=invalid,
        cross_product_citations=cross_product,
        rate=round(1 - len(invalid) / len(used), 3) if used else None,
        note="این نرخ فقط اعتبار شناسه و تعلق آن به محصول را بررسی می‌کند، نه صحت معنایی ادعا را.")
    return out

def compare_baseline(pids, question="کدام انتخاب بهتری است و چرا؟", model="openai/gpt-4.1-mini"):
    data = {int(p): facts(p) for p in pids}
    msg = [{"role": "system", "content": "فقط بر اساس جدول اعداد، محصولات را فارسی و کوتاه مقایسه کن."},
           {"role": "user", "content": f"سؤال: {question}\n" + json.dumps(data, ensure_ascii=False)}]
    return llm(msg, "compare_baseline", model=model, temperature=0).choices[0].message.content

print(json.dumps(compare_llm(pids), ensure_ascii=False, indent=1))
print("---BASELINE---"); print(compare_baseline(pids))
print(pd.DataFrame(LOG))

{
 "direct_facts": [
  {
   "product_id": 4415335,
   "claim": "قیمت 177000 تومان، امتیاز بیزی 3.85، تعداد نظرات نمونه 37، درصد پیشنهاد خرید 76.9%"
  },
  {
   "product_id": 1890184,
   "claim": "قیمت 900000 تومان، امتیاز بیزی 3.5، تعداد نظرات نمونه 12، درصد پیشنهاد خرید 90%"
  }
 ],
 "review_evidence": [
  {
   "product_id": 4415335,
   "claim": "کیف باکیفیت و خاص است، دوخت تمیز دارد، اما بند آن ضعیف به نظر می‌رسد و کیف دوشی نیست بلکه کوله است.",
   "comment_ids": [
    54199580,
    46688268,
    46669642,
    48736281,
    54583682
   ]
  },
  {
   "product_id": 1890184,
   "claim": "طراحی زیبا و ساخت خوب دارد، مناسب کاربری روزمره است، اما خیلی کوچک است و بند دوشی کیفیت پایینی دارد.",
   "comment_ids": [
    12716820,
    28714961,
    50274976,
    17361498
   ]
  }
 ],
 "inference": {
  "winner": null,
  "reasoning": "با توجه به فاصله معیارها یا تعداد شواهد، اعلام برنده قطعی قابل دفاع نیست.",
  "caveats": "تعداد نظرهای نمونه برای یکی از دو محصول کمتر از ۲۰ است؛ بیشترین قیمت 5.1 بر

In [9]:
TOOLS = [
 {"type": "function", "function": {"name": "category_complaints",
   "description": "پرتکرارترین شکایت‌ها و ویژگی‌هایی که کاربران یک دسته از آن‌ها ناراضی‌اند؛ همراه با سهم دسته‌بندی‌نشده و شناسه نمونه‌ها",
   "parameters": {"type": "object", "properties": {"category": {"type": "string", "description": "نام دسته به فارسی"}}, "required": ["category"]}}},
 {"type": "function", "function": {"name": "low_recommend_popular",
   "description": "محصولات پرنظر یک دسته که درصد پیشنهاد خرید پایین دارند؛ همراه با آستانه‌های انتخاب",
   "parameters": {"type": "object", "properties": {"category": {"type": "string"}}, "required": ["category"]}}},
 {"type": "function", "function": {"name": "brand_comparison",
   "description": "مقایسه‌ی بازخورد کاربران درباره‌ی برندهای اصلی یک دسته، با تعداد نمونه و موضوعات شکایت",
   "parameters": {"type": "object", "properties": {"category": {"type": "string"}}, "required": ["category"]}}},
]
FUNCS = dict(category_complaints=category_complaints, low_recommend_popular=low_recommend_popular,
             brand_comparison=brand_comparison)

SYS_MGR = """تو تحلیلگر داده برای مدیر یک دسته‌ی محصول در فروشگاه اینترنتی هستی.
حتماً از ابزارها استفاده کن و فقط از خروجی ابزار حرف بزن؛ عددها را عیناً بیاور و چیزی نساز.
برای شکایت‌های مهم comment_idهای نمونه را بیاور.
سهم نظرات دسته‌بندی‌نشده و محدودیت نمونه‌گیری را در صورت وجود گزارش کن.
تفاوت تعداد نمونه‌ی امتیازها و وضعیت پیشنهاد خرید را در مقایسه برندها در نظر بگیر.
اگر ابزار خطا داد همان را بگو و دسته‌ی دیگری را خودسرانه جایگزین نکن.
پاسخ فارسی، کوتاه و تصمیم‌محور باشد و آخرش ۲ تا ۳ پیشنهاد اقدام بدهد."""

jd = lambda o: json.dumps(o, ensure_ascii=False, default=lambda x: x.item() if hasattr(x, "item") else str(x))

def manager_ask(question, model="openai/gpt-4.1-mini"):
    msgs = [{"role": "system", "content": SYS_MGR}, {"role": "user", "content": question}]
    m = llm(msgs, "mgr_route", model=model, tools=TOOLS, temperature=0).choices[0].message
    if not m.tool_calls:
        return m.content, []
    msgs.append(m)
    called = []
    for tc in m.tool_calls:
        name = tc.function.name
        if name not in FUNCS:
            tool_result = {"error": f"ابزار ناشناخته: {name}"}
        else:
            args = json.loads(tc.function.arguments)
            called.append((name, args))
            try:
                tool_result = FUNCS[name](**args)
            except Exception as exc:
                tool_result = {"error": f"اجرای ابزار ناموفق بود: {type(exc).__name__}: {exc}"}
        msgs.append({"role": "tool", "tool_call_id": tc.id, "content": jd(tool_result)})
    answer = llm(msgs, "mgr_answer", model=model, temperature=0).choices[0].message.content
    return answer, called

def manager_ask_rules(question):
    qn = fa_norm(question)
    if any(w in qn for w in ["برند", "برندها"]):
        fn = brand_comparison
    elif any(w in qn for w in ["پرنظر", "پیشنهاد خرید پایین", "کمتر پیشنهاد"]):
        fn = low_recommend_popular
    else:
        fn = category_complaints
    return fn.__name__, fn(question)

print(manager_ask_rules(f"پرتکرارترین شکایت کاربران در دسته‌ی {CAT} چیست؟")[0])
ans, called = manager_ask(f"پرتکرارترین شکایت کاربران در دسته‌ی {CAT} چیست؟")
print(called); print(ans)

category_complaints
[('category_complaints', {'category': 'فکری و آموزشی'})]
شکایت پرتکرار کاربران در دسته‌ی فکری و آموزشی مربوط به "کیفیت و دوام" است با 7153 مورد (39.9٪ از شکایات منفی). نمونه comment_idهای مهم: 31999275، 34964006، 12103286.

عبارات پرتکرار شکایت شامل "کیفیت پایین" (89 بار، نمونه comment_id: 18523229)، "بی کیفیت" (79 بار، نمونه comment_id: 22020236) و "قیمت بالا" (42 بار، نمونه comment_id: 33276025) هستند.

سهم نظرات دسته‌بندی‌نشده 35.5٪ است که ممکن است محدودیت نمونه‌گیری ایجاد کند.

پیشنهادات:
1. بررسی و بهبود کیفیت محصولات با تمرکز بر دوام.
2. بازنگری قیمت‌گذاری برای افزایش ارزش خرید.
3. بهبود بسته‌بندی و خدمات پس از فروش جهت کاهش شکایات.


In [10]:
# چارچوب ارزیابی دستی و بازیابی
# این سلول به‌صورت پیش‌فرض هیچ APIای را فراخوانی نمی‌کند.
# سؤال‌ها و شناسه‌های مرتبط را پس از بازبینی انسانی در EVAL_CASES وارد کنید.

EVAL_CASES = [
    # نمونه ساختار:
    # {"question": "...", "task": "comparison",
    #  "relevant_comment_ids": [101, 202],
    #  "human_relevance": 1, "factual_correctness": 1,
    #  "grounded": 1, "usefulness": 1}
]

def retrieval_metrics(retrieved_ids, relevant_ids, k=5):
    """محاسبه Precision@k، Recall@k، F1@k و Reciprocal Rank."""
    retrieved = list(dict.fromkeys(map(str, retrieved_ids)))[:k]
    relevant = set(map(str, relevant_ids))
    if not relevant:
        return {"precision_at_k": None, "recall_at_k": None,
                "f1_at_k": None, "reciprocal_rank": None}
    hits = [i for i, cid in enumerate(retrieved, start=1) if cid in relevant]
    precision = len(hits) / k
    recall = len(hits) / len(relevant)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    rr = 1 / hits[0] if hits else 0.0
    return {"precision_at_k": round(precision, 3),
            "recall_at_k": round(recall, 3),
            "f1_at_k": round(f1, 3),
            "reciprocal_rank": round(rr, 3)}

def summarize_human_eval(eval_df):
    """خلاصه امتیازهای انسانی؛ ستون‌ها باید از قبل توسط ارزیاب تکمیل شده باشند."""
    score_cols = ["human_relevance", "factual_correctness", "grounded", "usefulness"]
    missing = [c for c in score_cols if c not in eval_df.columns]
    if missing:
        raise ValueError(f"ستون‌های ارزیابی موجود نیستند: {missing}")
    return eval_df[score_cols].apply(pd.to_numeric, errors="coerce").mean().round(3).to_dict()

# پس از تکمیل دستی:
# eval_df = pd.DataFrame(EVAL_CASES)
# print(summarize_human_eval(eval_df))


In [11]:
# 1) Reranking محصولات و شواهد نظرات با Cross-Encoder
# اجرای این سلول فقط توابع را تعریف می‌کند و مدل را دانلود/بارگذاری نمی‌کند.

_CROSS_ENCODER = None

def get_reranker(model_name="BAAI/bge-reranker-v2-m3", device="cpu"):
    """بارگذاری تنبل مدل؛ در اولین فراخوانی ممکن است مدل از اینترنت دانلود شود."""
    global _CROSS_ENCODER
    if _CROSS_ENCODER is None:
        try:
            from sentence_transformers import CrossEncoder
        except ImportError as exc:
            raise ImportError("ابتدا اجرا کنید: %pip install -q sentence-transformers") from exc
        _CROSS_ENCODER = CrossEncoder(model_name, max_length=512, device=device)
    return _CROSS_ENCODER

def _product_text(row):
    cols = ["title_fa", "Brand", "Category1", "Category2"]
    return " | ".join(str(row[c]) for c in cols if c in row and pd.notna(row[c]))

def rerank_products(query, candidate_k=50, top_k=10, model_name="BAAI/bge-reranker-v2-m3"):
    """ابتدا بازیابی Hybrid فعلی، سپس مرتب‌سازی مجدد همان کاندیدها."""
    candidates = eng.search(query, k=candidate_k, mode="hybrid", parse=True)
    if candidates is None or candidates.empty:
        return pd.DataFrame()
    model = get_reranker(model_name)
    pairs = [[str(query), _product_text(row)] for _, row in candidates.iterrows()]
    scores = model.predict(pairs)
    out = candidates.copy()
    out["rerank_score"] = np.asarray(scores, dtype=float)
    return out.sort_values("rerank_score", ascending=False).head(top_k).reset_index(drop=True)

def rerank_reviews(product_id, question, candidate_k=30, top_k=8,
                   intent="general", model_name="BAAI/bge-reranker-v2-m3"):
    """بازچینی شواهد بازیابی‌شده برای یک محصول؛ شناسه و متن نظر حفظ می‌شود."""
    evidence = rs.evidence(int(product_id), question, intent, k=candidate_k)
    if not evidence:
        return []
    model = get_reranker(model_name)
    pairs = [[str(question), str(item["text"])] for item in evidence]
    scores = np.asarray(model.predict(pairs), dtype=float)
    ranked = []
    for item, score in zip(evidence, scores):
        ranked.append({**item, "rerank_score": round(float(score), 4)})
    return sorted(ranked, key=lambda x: x["rerank_score"], reverse=True)[:top_k]

# نمونه اجرا پس از نصب وابستگی (اختیاری):
# rerank_products("کیف دوشی زنانه", candidate_k=40, top_k=10)
# rerank_reviews(int(pids[0]), "کیفیت و دوام محصول چطور است؟", intent="cons")


In [12]:
# 2) ارزیابی منصفانه‌ی reranker روی مجموعه‌ی کاندیدهای یکسان
# شناسه‌های مرتبط را باید با بازبینی انسانی وارد کنید؛ عدد ساختگی برای ارزیابی معتبر نیست.

RERANK_EVAL_CASES = [
    # {"product_id": 123, "question": "کیفیت محصول چطور است؟",
    #  "intent": "cons", "relevant_comment_ids": [456, 789]}
]

def evaluate_review_reranker(cases, candidate_k=30, top_k=8):
    rows = []
    model = get_reranker()
    for case in cases:
        pid = int(case["product_id"])
        question = case["question"]
        intent = case.get("intent", "general")
        relevant = case.get("relevant_comment_ids", [])
        # baseline و reranker دقیقاً از یک مجموعه کاندید استفاده می‌کنند.
        baseline = rs.evidence(pid, question, intent, k=candidate_k)
        pairs = [[question, str(x["text"])] for x in baseline]
        scores = np.asarray(model.predict(pairs), dtype=float) if pairs else np.array([])
        reranked = [x for _, x in sorted(zip(scores, baseline), key=lambda z: z[0], reverse=True)]
        base_ids = [x["comment_id"] for x in baseline[:top_k]]
        rerank_ids = [x["comment_id"] for x in reranked[:top_k]]
        rows.append({"product_id": pid, "question": question, "method": "ReviewSearch",
                     **retrieval_metrics(base_ids, relevant, k=top_k)})
        rows.append({"product_id": pid, "question": question, "method": "CrossEncoder",
                     **retrieval_metrics(rerank_ids, relevant, k=top_k)})
    return pd.DataFrame(rows)

# پس از تکمیل RERANK_EVAL_CASES با شناسه‌های واقعاً مرتبط:
# rerank_eval_df = evaluate_review_reranker(RERANK_EVAL_CASES)
# display(rerank_eval_df.groupby("method")[["precision_at_k", "recall_at_k", "f1_at_k", "reciprocal_rank"]].mean())


In [13]:
# 3) بررسی اتصال شواهد خروجی مقایسه به نظرات واقعی
# ساختار با خروجی compare_products در همین نوت‌بوک سازگار است.

def validate_review_evidence(compare_result, packs):
    """هر comment_id ادعاشده باید در شواهد همان محصول وجود داشته باشد."""
    issues = []
    review_block = compare_result.get("review_evidence", {})
    for pid_key, block in review_block.items():
        pid = int(pid_key)
        allowed = {int(x["comment_id"]) for x in packs.get(pid, {}).get("evidence", [])}
        for ev in block.get("evidence", []):
            cid = int(ev.get("comment_id", -1))
            if cid not in allowed:
                issues.append({"product_id": pid, "comment_id": cid,
                               "problem": "شناسه در شواهد بازیابی‌شده‌ی همین محصول وجود ندارد"})
    return {"valid": len(issues) == 0, "issues": issues}

# نمونه:
# check = validate_review_evidence(res, packs)
# print(json.dumps(check, ensure_ascii=False, indent=2))


In [14]:
# 4) دسته‌بندی کمکی شکایت‌ها با شباهت معنایی
# دسته‌بندی قانون‌محور complaint_report همچنان مرجع پایه است؛ این تابع صرفاً پیشنهاد معنایی می‌دهد.

TAXO_DESCRIPTIONS = {
    "کیفیت و دوام": "خرابی، پارگی، شکستگی، کیفیت پایین و دوام کم کالا",
    "ظاهر و تفاوت با عکس": "تفاوت رنگ، ظاهر یا مشخصات محصول با عکس و توضیحات",
    "سایز و اندازه": "نامناسب بودن اندازه، سایز، بلندی، کوتاهی، تنگی یا گشادی",
    "بسته‌بندی و ارسال": "تاخیر ارسال، بسته‌بندی نامناسب یا آسیب دیدن در حمل",
    "اصالت و تقلبی": "تقلبی بودن کالا یا اصل نبودن و مغایرت با برند اصلی",
    "قیمت و ارزش خرید": "گران بودن و تناسب نداشتن قیمت با کیفیت و ارزش خرید",
    "عملکرد": "درست کار نکردن، نقص عملکرد، باتری، شارژ یا کارایی محصول",
    "گارانتی و خدمات": "مشکل گارانتی، خدمات پس از فروش، پشتیبانی و مرجوعی",
}
_TAXO_EMB = None

def semantic_complaint_suggestions(texts, threshold=0.35, top_k=2):
    """برای هر متن، نزدیک‌ترین موضوع شکایت را همراه امتیاز شباهت برمی‌گرداند.
    آستانه باید با داده‌ی برچسب‌خورده تنظیم شود؛ خروجی، برچسب قطعی نیست.
    """
    global _TAXO_EMB
    labels = list(TAXO_DESCRIPTIONS)
    if _TAXO_EMB is None:
        _TAXO_EMB = eng.model.encode([fa_norm(TAXO_DESCRIPTIONS[x]) for x in labels],
                                     normalize_embeddings=True, show_progress_bar=False)
    vectors = eng.model.encode([fa_norm(str(t)) for t in texts],
                               normalize_embeddings=True, show_progress_bar=False)
    sims = np.asarray(vectors) @ np.asarray(_TAXO_EMB).T
    output = []
    for text, row in zip(texts, sims):
        order = np.argsort(row)[::-1][:top_k]
        suggestions = [{"theme": labels[i], "similarity": round(float(row[i]), 3)}
                       for i in order if float(row[i]) >= threshold]
        output.append({"text": text, "suggestions": suggestions,
                       "unclassified": len(suggestions) == 0})
    return output

# نمونه:
# semantic_complaint_suggestions(["بعد از دو هفته خراب شد", "رنگش با عکس فرق داشت"])


In [15]:
# 5) کش اختیاری برای پاسخ‌های LLM
# فقط برای درخواست‌های یکسان با همان مدل و پارامترها؛ پاک‌کردن کش با clear_llm_cache انجام می‌شود.
from functools import lru_cache

@lru_cache(maxsize=128)
def _cached_llm_text(messages_json, tag, model, temperature, max_tokens):
    messages = json.loads(messages_json)
    kwargs = {"temperature": temperature}
    if max_tokens is not None:
        kwargs["max_tokens"] = max_tokens
    response = llm(messages, tag, model=model, **kwargs)
    return response.choices[0].message.content or ""

def cached_llm(messages, tag="cached_call", model="openai/gpt-4.1-mini",
               temperature=0, max_tokens=700):
    key = json.dumps(messages, ensure_ascii=False, sort_keys=True, default=str)
    return _cached_llm_text(key, tag, model, float(temperature),
                            None if max_tokens is None else int(max_tokens))

def clear_llm_cache():
    _cached_llm_text.cache_clear()

# پاسخ این تابع متن است، نه آبجکت کامل API:
# answer = cached_llm([{"role": "user", "content": "سلام"}])
# print(answer)


In [16]:
# 6) قالب ارزیابی انسانی و نمودار خلاصه
# بعد از اجرای سیستم، ارزیاب برای هر نمونه امتیاز 1 تا 5 وارد می‌کند.

HUMAN_EVAL_COLUMNS = ["question", "task", "human_relevance",
                      "factual_correctness", "grounded", "usefulness", "notes"]

def make_human_eval_template(questions):
    df = pd.DataFrame(questions)
    for col in HUMAN_EVAL_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan
    return df[HUMAN_EVAL_COLUMNS]

def summarize_human_eval_5pt(eval_df):
    score_cols = ["human_relevance", "factual_correctness", "grounded", "usefulness"]
    missing = [c for c in score_cols if c not in eval_df.columns]
    if missing:
        raise ValueError(f"ستون‌های ارزیابی موجود نیستند: {missing}")
    scores = eval_df[score_cols].apply(pd.to_numeric, errors="coerce")
    if ((scores < 1) | (scores > 5)).any().any():
        raise ValueError("امتیازهای ارزیابی باید بین 1 تا 5 باشند.")
    return scores.mean().round(2)

# مثال:
# human_df = make_human_eval_template([
#     {"question": "شکایت‌های رایج این دسته چیست؟", "task": "complaint"},
#     {"question": "کدام محصول بهتر است؟", "task": "comparison"}
# ])
# پس از تکمیل دستی ستون‌های امتیاز:
# display(summarize_human_eval_5pt(human_df))
# human_df.to_csv("human_evaluation.csv", index=False, encoding="utf-8-sig")


In [17]:
query = "کیف دوشی زنانه"

baseline = eng.search(
    query,
    k=10,
    mode="hybrid",
    parse=True
)

display(
    baseline[[
        c for c in ["id", "title_fa", "Brand", "Category2", "Price"]
        if c in baseline.columns
    ]]
)

,id,title_fa,Brand,Category2
0,4415335,کیف دوشی زنانه مدل BRUNCH,متفرقه,کیف زنانه
1,1890184,کیف دوشی زنانه طرح پروانه,متفرقه,کیف زنانه
2,10724796,کیف دوشی زنانه مدل 789,متفرقه,کیف زنانه
3,1977996,کیف دوشی زنانه مدل LAMIA,متفرقه,کیف زنانه
4,2190423,کیف دوشی زنانه کد 32355,متفرقه,کیف زنانه
5,3003173,کیف دوشی زنانه کد‌‌‌‌‌‌‌‌‌‌ 250,متفرقه,کیف زنانه
6,1102273,کیف دوشی زنانه گالری چی چاپ کد 66001,متفرقه,کیف زنانه
7,1639650,کیف دوشی زنانه گالری چی چاپ طرح پرستار,متفرقه,کیف زنانه
8,9998148,کیف دوشی زنانه مدل 280s,متفرقه,کیف زنانه
9,2541091,کیف دوشی زنانه کد 844,متفرقه,کیف زنانه


In [21]:
reranked = rerank_products(
    query,
    candidate_k=40,
    top_k=10
)

display(
    reranked[[
        c for c in [
            "id", "title_fa", "Brand",
            "Category2", "rerank_score"
        ]
        if c in reranked.columns
    ]]
)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 500.27it/s]


,id,title_fa,Brand,Category2,rerank_score
0,9651665,کیف دوشی زنانه مدل NU-sh00966,متفرقه,کیف زنانه,0.997587
1,7294241,کیف دوشی زنانه مدل NU-00670,متفرقه,کیف زنانه,0.997520
2,7005308,کیف دوشی زنانه مدل KS-0075,متفرقه,کیف زنانه,0.997475
3,9011582,کیف دوشی زنانه مدل NU-00876,متفرقه,کیف زنانه,0.997408
4,9307403,کیف دوشی زنانه مدل 15277,متفرقه,کیف زنانه,0.997381
5,9102881,کیف دوشی زنانه مدل NU-00898,متفرقه,کیف زنانه,0.997202
6,9224401,کیف دوشی زنانه مدل 39051,متفرقه,کیف زنانه,0.997179
7,7608416,کیف دوشی زنانه مدل A1600,متفرقه,کیف زنانه,0.996615
8,1977996,کیف دوشی زنانه مدل LAMIA,متفرقه,کیف زنانه,0.996596
9,8690228,کیف دوشی زنانه مدل N29,متفرقه,کیف زنانه,0.996380


In [19]:
import json
import re

JUDGE_SYSTEM = """
تو یک ارزیاب مستقل برای پاسخ‌های یک سیستم هوش مصنوعی فروشگاهی هستی.

پاسخ را بر اساس معیارهای زیر ارزیابی کن:

1. relevance: آیا پاسخ به سؤال کاربر مرتبط است؟ (1 تا 5)
2. correctness: آیا اطلاعات پاسخ با داده‌های مرجع سازگار است؟ (1 تا 5)
3. completeness: آیا پاسخ به اندازه کافی کامل است؟ (1 تا 5)
4. clarity: آیا پاسخ روان و قابل فهم است؟ (1 تا 5)
5. groundedness: آیا ادعاها به اطلاعات و شواهد داده‌شده متکی هستند؟ (1 تا 5)

قوانین:
- اگر داده مرجع وجود ندارد، صحت اطلاعات را قطعی تأیید نکن.
- اطلاعات ساختگی و ادعاهای بدون مدرک را مشخص کن.
- فقط به دلیل طولانی بودن، پاسخ را کامل ارزیابی نکن.
- اگر پاسخ از داده‌های مرجع فراتر رفته، آن را گزارش کن.
- پاسخ را به زبان فارسی و در قالب JSON معتبر برگردان.

ساختار خروجی:
{
  "scores": {
    "relevance": 1,
    "correctness": 1,
    "completeness": 1,
    "clarity": 1,
    "groundedness": 1
  },
  "overall_score": 1,
  "verdict": "خوب / متوسط / ضعیف",
  "strengths": [],
  "problems": [],
  "suggestions": [],
  "explanation": ""
}
"""

def evaluate_ai_answer(question, answer, reference_context=""):
    user_content = f"""
سؤال کاربر:
{question}

پاسخ تولیدشده توسط AI:
{answer}

اطلاعات مرجع سیستم:
{reference_context if reference_context else "اطلاعات مرجع ارائه نشده است."}

پاسخ را ارزیابی کن.
"""

    response = llm(
        [
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": user_content}
        ],
        "answer_quality_judge",
        temperature=0,
        max_tokens=1200
    )

    raw = response.choices[0].message.content

    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", raw, re.S)
        if not match:
            raise ValueError("خروجی ارزیاب JSON معتبر نیست:\n" + raw)
        result = json.loads(match.group())

    return result

In [20]:
question = f"پرتکرارترین شکایت کاربران در دسته {CAT} چیست؟"

# پاسخ واقعی تولیدشده توسط API
answer, called = manager_ask(question)

# اطلاعات مرجع که ابزارهای سیستم تولید کرده‌اند
reference = json.dumps(
    category_complaints(CAT),
    ensure_ascii=False,
    default=str
)

# ارزیابی پاسخ
evaluation = evaluate_ai_answer(
    question=question,
    answer=answer,
    reference_context=reference
)

print("===== AI ANSWER =====")
print(answer)

print("\n===== QUALITY EVALUATION =====")
print(json.dumps(
    evaluation,
    ensure_ascii=False,
    indent=2
))

===== AI ANSWER =====
پرتکرارترین شکایت کاربران در دسته فکری و آموزشی مربوط به "کیفیت و دوام" است که 39.9٪ از شکایت‌ها را شامل می‌شود (7153 مورد). نمونه comment_idهای این شکایت: 31999275، 34964006، 12103286.

سهم نظرات دسته‌بندی‌نشده 35.5٪ است که ممکن است تحلیل را محدود کند.

شکایات مهم دیگر شامل "ظاهر و تفاوت با عکس" (13.8٪)، "قیمت و ارزش خرید" (9.5٪)، و "بسته‌بندی و ارسال" (9.4٪) هستند.

پیشنهادات:
1. بررسی و بهبود کیفیت و دوام محصولات این دسته.
2. شفاف‌سازی تصاویر و توضیحات محصول برای کاهش تفاوت با عکس.
3. بازنگری قیمت‌گذاری و بسته‌بندی برای افزایش رضایت مشتری.

===== QUALITY EVALUATION =====
{
  "scores": {
    "relevance": 5,
    "correctness": 5,
    "completeness": 4,
    "clarity": 5,
    "groundedness": 5
  },
  "overall_score": 5,
  "verdict": "خوب",
  "strengths": [
    "پاسخ کاملاً مرتبط با سؤال است و پرتکرارترین شکایت را به درستی بیان کرده است.",
    "اطلاعات ارائه شده دقیقاً مطابق داده‌های مرجع است، از جمله درصدها و تعداد موارد شکایت.",
    "پاسخ به صورت واضح و روان نوشته

In [ ]:
#چه پروژه ای بود واقعا-----